# Molecular Interaction Explorer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sunil-Paliwal/molecular-interaction-explorer-colab/blob/main/Molecular_Interaction_Explorer_Colab.ipynb)

**Author:** Dr. Sunil Paliwal, Department of Chemistry and Chemical Biology, Stevens Institute of Technology  
**Source:** https://github.com/Sunil-Paliwal/molecular-interaction-explorer-colab

Enter two molecules as SMILES. The tool identifies the **strongest** non-covalent interaction the pair can form, draws the two molecules facing each other, and marks the interacting atoms.

| Tier | Interaction | What the tool looks for |
|---|---|---|
| 1 | Ionic / salt bridge | Opposite full charges (e.g. COO⁻ ··· NH₃⁺) |
| 2 | Ion–dipole | One ion + a polar neutral molecule |
| 3 | Hydrogen bond | O–H or N–H donor ··· O or N lone-pair acceptor |
| 4 | Dipole–dipole | Both molecules have a net dipole |
| 5 | van der Waals / dispersion | Nonpolar molecules, or polar bonds that cancel by symmetry |

**Note:** real molecules usually make several interactions at once; this tool shows only the strongest one. Charges in neutral groups drawn as N⁺/O⁻ (nitro, N-oxide) are correctly treated as neutral.

**Try:** `CC(=O)[O-]` + `CNC(=[NH2+])N` · `[Na+]` + `O` · `CCO` + `CC(C)=O` · `CC(C)=O` + `CC(C)=O` · `CCCC` + `CCCC` · `O=C=O` + `O=C=O` · `C[NH3+]` + `C[NH3+]`

Run the two code cells below in order.

In [ ]:
# Step 1: Install dependencies (takes ~30 s)
!pip install -q rdkit

In [ ]:
# Step 2: Load the tool and show the input boxes
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D

RDLogger.DisableLog('rdApp.*')  # keep student output clean; we print our own errors


# =========================================================
# Interaction styles (tier number, label, color)
# =========================================================
TIERS = {
    1: ("Tier 1: Ionic Interaction (Salt Bridge)", "#E67E22"),
    2: ("Tier 2: Ion-Dipole Interaction", "#8E44AD"),
    3: ("Tier 3: Hydrogen Bond", "#C0392B"),
    4: ("Tier 4: Dipole-Dipole Interaction", "#27AE60"),
    5: ("Tier 5: van der Waals / Dispersion", "#7F8C8D"),
    0: ("No Attraction: Like Charges Repel", "#566573"),
}

DIPOLE_THRESHOLD_D = 0.2   # below this a molecule is treated as nonpolar (e.g. CO2, CCl4)


# =========================================================
# Chemistry helpers
# =========================================================
def sanitize_and_depict(mol):
    """Returns a copy with clean 2D coordinates."""
    m = Chem.Mol(mol)
    Chem.SanitizeMol(m)
    rdDepictor.Compute2DCoords(m)
    return m


def ionic_atoms(mol):
    """
    Atoms carrying a *real* ionic charge.
    Charges that are cancelled by an oppositely charged bonded neighbour
    (nitro N+/O-, N-oxides, azides, C[S+](C)[O-] ...) are just a way of drawing a
    neutral group, so they are ignored. Separated charges (zwitterions) are kept.
    """
    out = []
    for a in mol.GetAtoms():
        q = a.GetFormalCharge()
        if q == 0:
            continue
        if any(n.GetFormalCharge() * q < 0 for n in a.GetNeighbors()):
            continue
        out.append((a.GetIdx(), q))
    return out


def gasteiger_heavy(mol):
    """Gasteiger partial charges of heavy atoms, with implicit H charge folded in."""
    mh = Chem.AddHs(mol)
    AllChem.ComputeGasteigerCharges(mh)
    charges = {}
    for a in mh.GetAtoms():
        if a.GetAtomicNum() == 1:
            continue
        q = a.GetDoubleProp('_GasteigerCharge')
        q += sum(n.GetDoubleProp('_GasteigerCharge') for n in a.GetNeighbors() if n.GetAtomicNum() == 1)
        charges[a.GetIdx()] = q if np.isfinite(q) else 0.0
    return charges


def dipole_debye(mol):
    """Approximate molecular dipole (Debye) from a 3D conformer + Gasteiger charges."""
    mh = Chem.AddHs(mol)
    try:
        AllChem.ComputeGasteigerCharges(mh)
        params = AllChem.ETKDGv3()
        params.randomSeed = 42
        if AllChem.EmbedMolecule(mh, params) != 0:
            return None
        if AllChem.MMFFHasAllMoleculeParams(mh):
            AllChem.MMFFOptimizeMolecule(mh)
        pos = mh.GetConformer().GetPositions()
        q = np.array([a.GetDoubleProp('_GasteigerCharge') for a in mh.GetAtoms()])
        if not np.all(np.isfinite(q)):
            return None
        return float(np.linalg.norm((q[:, None] * pos).sum(axis=0)) * 4.803)  # e*A -> Debye
    except Exception:
        return None


def is_polar(mol):
    """Polar = has a net dipole. Falls back to 'contains O/N/F/Cl/Br/S' if 3D fails."""
    mu = dipole_debye(mol)
    if mu is None:
        return mol.HasSubstructMatch(Chem.MolFromSmarts("[O,N,F,Cl,Br,S]")), None
    return mu > DIPOLE_THRESHOLD_D, mu


HBD_LIST = [Chem.MolFromSmarts("[O;!H0;+0]"),                     # O-H donors (strongest)
            Chem.MolFromSmarts("[N,n;!H0;+0]")]                   # N-H donors
HBA_LIST = [                                                      # acceptors, best first
    Chem.MolFromSmarts("[OX1;+0]=[#6,#16,#15]"),                  # C=O, S=O, P=O oxygen
    Chem.MolFromSmarts("[nX2;+0;H0]"),                            # pyridine-type N
    Chem.MolFromSmarts("[NX3;+0;!$(N-[C,S,P]=[O,S]);!$(N-a);!$(N-[N,O])]"),  # amine N
    Chem.MolFromSmarts("[OX2;+0]"),                               # alcohol / ether / water O
    Chem.MolFromSmarts("[NX2;+0]=*"),                             # imine N
    Chem.MolFromSmarts("[NX1;+0]#*"),                             # nitrile N
]


def first_match(mol, patt):
    m = mol.GetSubstructMatches(patt)
    return m[0][0] if m else None


def best_donor(mol):
    for patt in HBD_LIST:
        idx = first_match(mol, patt)
        if idx is not None:
            return idx
    return None


def best_acceptor(mol):
    for patt in HBA_LIST:
        idx = first_match(mol, patt)
        if idx is not None:
            return idx
    return None


def atom_label(mol, idx):
    a = mol.GetAtomWithIdx(idx)
    s = a.GetSymbol()
    h = a.GetTotalNumHs()
    if h and s != 'C':
        hs = 'H' + (str(h) if h > 1 else '')
        s = hs + s if a.GetDegree() == 0 else s + hs     # water -> H2O, alcohol -> OH
    q = a.GetFormalCharge()
    if q:
        s += ('+' if q > 0 else '−') if abs(q) == 1 else f"{abs(q)}{'+' if q > 0 else '−'}"
    return s


# =========================================================
# Classifier: returns the STRONGEST interaction the pair can make
# =========================================================
def find_interaction_centers(mol_a, mol_b):
    """
    Returns (idx_a, idx_b, tier, label, color, explanation).
    Hierarchy: 1 ionic > 2 ion-dipole > 3 H-bond > 4 dipole-dipole > 5 dispersion.
    """
    ions_a, ions_b = ionic_atoms(mol_a), ionic_atoms(mol_b)

    def done(ia, ib, tier, why, roles=("", "")):
        label, color = TIERS[tier]
        return ia, ib, tier, label, color, why, roles

    # ---- Tier 1: opposite full charges ----
    neg_a = [i for i, q in ions_a if q < 0]; pos_a = [i for i, q in ions_a if q > 0]
    neg_b = [i for i, q in ions_b if q < 0]; pos_b = [i for i, q in ions_b if q > 0]
    if neg_a and pos_b:
        return done(neg_a[0], pos_b[0], 1, "Full negative charge attracts full positive charge.")
    if pos_a and neg_b:
        return done(pos_a[0], neg_b[0], 1, "Full positive charge attracts full negative charge.")

    # ---- Both ions, same sign: repulsive ----
    if ions_a and ions_b:
        return done(ions_a[0][0], ions_b[0][0], 0,
                    "Both species carry the same charge, so they repel instead of binding.")

    # ---- Tier 2: one ion + a neutral partner ----
    if ions_a or ions_b:
        ion_mol, ions, partner = (mol_a, ions_a, mol_b) if ions_a else (mol_b, ions_b, mol_a)
        ion_idx, ion_q = ions[0]
        polar, _ = is_polar(partner)
        q = gasteiger_heavy(partner)
        if ion_q > 0:
            # cation -> most electron-rich (δ−) atom of partner
            tgt = best_acceptor(partner)
            if tgt is None:
                tgt = min(q, key=q.get)
            why = "Cation is attracted to the δ− end of the neutral molecule's dipole."
        else:
            # anion -> δ+ end: an X–H donor if present, otherwise the most δ+ atom
            tgt = best_donor(partner)
            if tgt is not None:
                why = "Anion is attracted to the δ+ X–H of the neutral molecule (charge-assisted H-bond)."
            else:
                tgt = max(q, key=q.get)
                why = "Anion is attracted to the δ+ end of the neutral molecule's dipole."
        tier = 2
        if not polar and partner.GetNumAtoms() > 1:
            tier = 5
            why = "Partner is nonpolar: the ion only induces a weak temporary dipole (ion–induced dipole)."
        role = ("δ−" if ion_q > 0 else "δ+ H") if tier == 2 else ""
        ia, ib = (ion_idx, tgt) if ions_a else (tgt, ion_idx)
        roles = ("", role) if ions_a else (role, "")
        return done(ia, ib, tier, why, roles)

    # ---- Tier 3: hydrogen bond (donor X–H ... acceptor O/N) ----
    don_a, acc_b = best_donor(mol_a), best_acceptor(mol_b)
    if don_a is not None and acc_b is not None:
        return done(don_a, acc_b, 3, "X–H donor (molecule 1) shares its H with a lone pair on the acceptor (molecule 2).",
                    ("donor, Hδ+", "acceptor, δ−"))
    don_b, acc_a = best_donor(mol_b), best_acceptor(mol_a)
    if don_b is not None and acc_a is not None:
        return done(acc_a, don_b, 3, "X–H donor (molecule 2) shares its H with a lone pair on the acceptor (molecule 1).",
                    ("acceptor, δ−", "donor, Hδ+"))

    # ---- Tier 4: dipole-dipole (both molecules need a NET dipole) ----
    polar_a, mu_a = is_polar(mol_a)
    polar_b, mu_b = is_polar(mol_b)
    qa, qb = gasteiger_heavy(mol_a), gasteiger_heavy(mol_b)
    if polar_a and polar_b:
        a_neg, a_pos = min(qa, key=qa.get), max(qa, key=qa.get)
        b_neg, b_pos = min(qb, key=qb.get), max(qb, key=qb.get)
        if (qb[b_pos] - qa[a_neg]) >= (qa[a_pos] - qb[b_neg]):
            return done(a_neg, b_pos, 4, "δ− end of one permanent dipole lines up with the δ+ end of the other.", ("δ−", "δ+"))
        return done(a_pos, b_neg, 4, "δ+ end of one permanent dipole lines up with the δ− end of the other.", ("δ+", "δ−"))

    # ---- Tier 5: dispersion (incl. dipole–induced dipole) ----
    if polar_a != polar_b:
        why = "Only one molecule is polar: its dipole induces a weak dipole in the other (plus dispersion)."
        ia = min(qa, key=qa.get) if polar_a else mol_a.GetNumAtoms() // 2
        ib = min(qb, key=qb.get) if polar_b else mol_b.GetNumAtoms() // 2
        return done(ia, ib, 5, why)
    why = "No net dipoles: only temporary, induced dipoles (London dispersion) hold them together."
    if mol_a.HasSubstructMatch(Chem.MolFromSmarts("[O,N,F,Cl,Br,S]")):
        why = "Polar bonds cancel by symmetry (no net dipole), so only dispersion remains."
    return done(mol_a.GetNumAtoms() // 2, mol_b.GetNumAtoms() // 2, 5, why)


# =========================================================
# 2D layout helpers
# =========================================================
def _xy(conf, n):
    return np.array([[conf.GetAtomPosition(i).x, conf.GetAtomPosition(i).y] for i in range(n)])


def _set_xy(conf, xy):
    for i, (x, y) in enumerate(xy):
        conf.SetAtomPosition(i, (float(x), float(y), 0.0))


def outward_vector(mol, xy, idx):
    """Direction pointing 'out' of the molecule from atom idx."""
    a = mol.GetAtomWithIdx(idx)
    nbrs = [n.GetIdx() for n in a.GetNeighbors()]
    if nbrs:
        v = xy[idx] - xy[nbrs].mean(axis=0)
        if np.linalg.norm(v) < 0.3:                      # e.g. central carbonyl C
            heavy = max(nbrs, key=lambda j: mol.GetAtomWithIdx(j).GetAtomicNum())
            v = xy[idx] - xy[heavy]
    else:
        v = xy[idx] - xy.mean(axis=0)
    n = np.linalg.norm(v)
    return v / n if n > 1e-6 else None


def orient(mol, idx, target_angle):
    """Rotate molecule so the interacting atom points along target_angle."""
    conf = mol.GetConformer()
    xy = _xy(conf, mol.GetNumAtoms())
    v = outward_vector(mol, xy, idx)
    if v is None:
        return
    ang = target_angle - np.arctan2(v[1], v[0])
    R = np.array([[np.cos(ang), -np.sin(ang)], [np.sin(ang), np.cos(ang)]])
    c = xy[idx]
    _set_xy(conf, (xy - c) @ R.T + c)


def _pca_align(xy):
    """Center a 2D structure and turn its long axis horizontal."""
    c = xy - xy.mean(axis=0)
    if len(c) < 2:
        return c
    _, vecs = np.linalg.eigh(np.cov(c.T))
    main = vecs[:, -1]
    ang = -np.arctan2(main[1], main[0])
    R = np.array([[np.cos(ang), -np.sin(ang)], [np.sin(ang), np.cos(ang)]])
    return c @ R.T


def layout_surface_contact(mol_a, mol_b, gap=2.4, contact_extra=1.0, max_lines=6):
    """
    Dispersion has no single 'key atom': it comes from many weak contacts across the
    surfaces that touch. Try many orientations/offsets of the two molecules, rest B on top
    of A with the closest atoms `gap` apart, and keep the arrangement with the broadest
    contact. Long molecules stay horizontal; compact ones (rings) may rotate.
    Returns a list of (atom_in_A, atom_in_B) contact pairs.
    """
    conf_a, conf_b = mol_a.GetConformer(), mol_b.GetConformer()
    xa0 = _pca_align(_xy(conf_a, mol_a.GetNumAtoms()))
    xb0 = _pca_align(_xy(conf_b, mol_b.GetNumAtoms()))

    def angles(mol, xy):
        if len(xy) < 2:
            return [0.0]
        ev = np.linalg.eigvalsh(np.cov(xy.T))
        if ev[0] < 1e-6 or ev[1] / max(ev[0], 1e-6) > 2.5:
            return [0.0, np.pi]                       # long molecule: keep it horizontal
        # compact molecule (e.g. a ring): try every rotation that lays a bond flat
        out = set()
        for b in mol.GetBonds():
            v = xy[b.GetEndAtomIdx()] - xy[b.GetBeginAtomIdx()]
            t = -np.arctan2(v[1], v[0])
            for tt in (t, t + np.pi):
                out.add(round(tt % (2 * np.pi), 2))
        return sorted(out)

    def rot(xy, t):
        R = np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])
        return xy @ R.T

    def rest_on_top(xa, xb):
        """Smallest upward shift of B so every A-B distance is >= gap."""
        dx = xa[:, None, 0] - xb[None, :, 0]
        dy = xa[:, None, 1] - xb[None, :, 1]
        close = np.abs(dx) < gap
        need = np.where(close, dy + np.sqrt(np.clip(gap**2 - dx**2, 0, None)), -np.inf)
        return max(need.max(), xa[:, 1].max() - xb[:, 1].min() - gap)

    half_w = 0.5 * max(np.ptp(xa0[:, 0]) if len(xa0) > 1 else 0, np.ptp(xb0[:, 0]) if len(xb0) > 1 else 0)
    offsets = np.linspace(-0.4, 0.4, 5) * half_w if half_w > 0 else [0.0]

    best = None
    for ta in angles(mol_a, xa0):
        xa = rot(xa0, ta)
        for tb in angles(mol_b, xb0):
            for mirror in (1, -1):
                xb = rot(xb0 * [mirror, 1], tb)
                for ox in offsets:
                    xb_s = xb + [ox, 0]
                    xb_s = xb_s + [0, rest_on_top(xa, xb_s)]
                    D = np.linalg.norm(xa[:, None, :] - xb_s[None, :, :], axis=2)
                    # contact quality: close AND roughly vertical (molecules lying parallel)
                    vy = np.abs(xb_s[None, :, 1] - xa[:, None, 1]) / np.maximum(D, 1e-6)
                    w = np.clip(1 - (D - gap) / contact_extra, 0, None) * vy**2
                    # count each atom in at most one contact (one-to-one pairs)
                    score, ua, ub = 0.0, set(), set()
                    for flat in np.argsort(-w, axis=None):
                        i, j = np.unravel_index(flat, w.shape)
                        if w[i, j] <= 0:
                            break
                        if i in ua or j in ub:
                            continue
                        score += w[i, j]; ua.add(i); ub.add(j)
                    # tiny bias toward the plain drawing and centred stacking
                    score -= 0.01 * (ta != 0) + 0.01 * abs(ox)
                    if best is None or score > best[0] + 1e-9:
                        best = (score, xa, xb_s, D)

    _, xa, xb, D = best
    _set_xy(conf_a, xa)
    _set_xy(conf_b, xb)

    pairs, used_a, used_b = [], set(), set()
    for flat in np.argsort(D, axis=None):
        ia, ib = np.unravel_index(flat, D.shape)
        if D[ia, ib] > gap + contact_extra and pairs:
            break
        if ia in used_a or ib in used_b:
            continue
        pairs.append((int(ia), int(ib)))
        used_a.add(ia); used_b.add(ib)
        if len(pairs) >= max_lines:
            break
    return pairs


# =========================================================
# Main plotting routine
# =========================================================
def classify_and_plot_interaction(smiles_a: str, smiles_b: str):
    raw_a = Chem.MolFromSmiles(smiles_a.strip())
    raw_b = Chem.MolFromSmiles(smiles_b.strip())
    if raw_a is None or raw_b is None:
        bad = [s for s, m in ((smiles_a, raw_a), (smiles_b, raw_b)) if m is None]
        print("Error: invalid SMILES: " + ", ".join(repr(s) for s in bad))
        return

    mol_a = sanitize_and_depict(raw_a)
    mol_b = sanitize_and_depict(raw_b)
    idx_a, idx_b, tier, label, color, why, roles = find_interaction_centers(mol_a, mol_b)

    na, nb = mol_a.GetNumAtoms(), mol_b.GetNumAtoms()
    conf_a, conf_b = mol_a.GetConformer(), mol_b.GetConformer()

    contacts = []
    if tier == 5:
        # Broad surface contact: many weak contacts, not one atom pair
        contacts = layout_surface_contact(mol_a, mol_b)
    else:
        # Face-to-face: A's atom points right, B's atom points left, then align
        orient(mol_a, idx_a, 0.0)
        orient(mol_b, idx_b, np.pi)
        xa, xb = _xy(conf_a, na), _xy(conf_b, nb)
        gap = 3.6
        x_shift = max(xa[idx_a, 0] + gap - xb[idx_b, 0],      # target distance
                      xa[:, 0].max() + 1.8 - xb[:, 0].min())  # never overlap
        y_shift = xa[idx_a, 1] - xb[idx_b, 1]
        _set_xy(conf_b, xb + np.array([x_shift, y_shift]))

    combined = Chem.CombineMols(mol_a, mol_b)
    idx_b_comb = na + idx_b

    if tier == 5:
        # Lightly highlight every atom that takes part in a surface contact
        hl = {i: (0.86, 0.86, 0.86) for i, _ in contacts}
        hl.update({na + j: (0.86, 0.86, 0.86) for _, j in contacts})
        for ion_mol, off in ((mol_a, 0), (mol_b, na)):      # ions keep their colour
            for i, q in ionic_atoms(ion_mol):
                hl[off + i] = (1.0, 0.75, 0.75) if q < 0 else (0.75, 0.82, 1.0)
        line_pairs = [(i, na + j) for i, j in contacts]
    else:
        # Highlight the two interacting atoms
        hl = {idx_a: (1.0, 0.75, 0.75), idx_b_comb: (0.75, 0.82, 1.0)}
        line_pairs = [(idx_a, idx_b_comb)]

    drawer = rdMolDraw2D.MolDraw2DCairo(900, 420)
    opts = drawer.drawOptions()
    opts.useBWAtomPalette()
    opts.explicitMethyl = True
    opts.padding = 0.20
    opts.bondLineWidth = 2
    opts.fixedBondLength = 45          # consistent size: tiny molecules don't blow up
    drawer.DrawMolecule(combined, highlightAtoms=list(hl), highlightAtomColors=hl,
                        highlightBonds=[])
    line_pts = [(drawer.GetDrawCoords(i), drawer.GetDrawCoords(j)) for i, j in line_pairs]
    drawer.FinishDrawing()

    img = Image.open(io.BytesIO(drawer.GetDrawingText()))
    fig, ax = plt.subplots(figsize=(9, 4.8), dpi=120)
    ax.imshow(img)
    ax.axis('off')

    def atom_pad(idx):
        a = combined.GetAtomWithIdx(idx)
        return 30.0 if (a.GetTotalNumHs() and a.GetSymbol() != 'C') or a.GetFormalCharge() else 20.0

    for (p1, p2), (i, j) in zip(line_pts, line_pairs):
        pt1, pt2 = np.array([p1.x, p1.y]), np.array([p2.x, p2.y])
        d = np.linalg.norm(pt2 - pt1)
        if d == 0:
            continue
        u = (pt2 - pt1) / d
        pad1, pad2 = atom_pad(i), atom_pad(j)
        if tier == 5:
            pad1 = pad2 = 12.0
        if pad1 + pad2 > 0.8 * d:                       # keep the line visible when atoms are close
            pad1 = pad2 = 0.2 * d
        s_, e_ = pt1 + u * pad1, pt2 - u * pad2
        if tier == 5:
            ax.plot([s_[0], e_[0]], [s_[1], e_[1]], color=color, linestyle=(0, (1, 3)),
                    linewidth=2.0, alpha=0.8, dash_capstyle='round')
        else:
            style = ':' if tier == 0 else '--'
            ax.plot([s_[0], e_[0]], [s_[1], e_[1]], color=color, linestyle=style,
                    linewidth=2.8, dash_capstyle='round')

    def tag(mol, idx, role):
        lab = atom_label(mol, idx)
        return f"{lab} ({role})" if role else lab
    if tier == 5:
        ions = [atom_label(m, i) for m in (mol_a, mol_b) for i, _ in ionic_atoms(m)]
        pair = f"{len(contacts)} close contact(s) across the facing surfaces"
        if ions:
            pair = f"{ions[0]} induces a temporary dipole in its neighbour ({pair})"
    else:
        pair = f"Molecule 1: {tag(mol_a, idx_a, roles[0])}   ···   Molecule 2: {tag(mol_b, idx_b, roles[1])}"
    ax.set_title(label, fontsize=15, color=color, fontweight='bold', pad=10)
    fig.text(0.5, 0.10, pair, ha='center', fontsize=11, color='#333333')
    fig.text(0.5, 0.05, why, ha='center', fontsize=10, color='#555555', style='italic')
    plt.tight_layout(rect=(0, 0.12, 1, 1))
    plt.show()


# =========================================================
# Interactive UI
# =========================================================
import ipywidgets as widgets
from IPython.display import display, clear_output

text_a = widgets.Text(value='', description='SMILES 1:',
                      placeholder='e.g. CC(=O)[O-]  (acetate)',
                      style={'description_width': 'initial'}, layout=widgets.Layout(width='480px'))
text_b = widgets.Text(value='', description='SMILES 2:',
                      placeholder='e.g. CNC(=[NH2+])N  (methylguanidinium)',
                      style={'description_width': 'initial'}, layout=widgets.Layout(width='480px'))
button = widgets.Button(description='Analyze Interaction', button_style='danger',
                        icon='flask', layout=widgets.Layout(width='220px'))
output = widgets.Output()

def on_button_clicked(_):
    with output:
        clear_output(wait=True)
        if not text_a.value.strip() or not text_b.value.strip():
            print("Please enter a SMILES string in both fields.")
            return
        try:
            classify_and_plot_interaction(text_a.value, text_b.value)
        except Exception as e:
            print(f"Could not analyze this pair: {e}")

button.on_click(on_button_clicked)
display(widgets.VBox([text_a, text_b, button]), output)
